In [ ]:
import sys
sys.path.append('..')

from Combined.DataPrep.Data_Prep import Combined_Data_Prep
from Combined.DataPrep.Player_Dataset import Create_Test_Train_Datasets
from Pro.DataPrep import Prep_Map, Output_Map
from College.DataPrep import Prep_Map as C_Prep_Map, Output_Map as C_Output_Map

In [ ]:
data_prep = Combined_Data_Prep(
    Prep_Map.base_prep_map, 
    Output_Map.base_output_map,
    C_Prep_Map.college_base_prep_map,
    C_Output_Map.college_output_map)

hitter_io_list = data_prep.Generate_IO_Hitters(pro_player_condition="WHERE lastMLBSeason<? AND signingYear<? AND isHitter=?", pro_player_values=(2025,2015,1), pro_use_cutoff=True,
                                        col_player_condition="WHERE LastYear<=? AND isHitter=?", col_player_values=(2015, 1), col_use_cutoff=True)


In [ ]:
train_dataset, test_dataset = Create_Test_Train_Datasets(hitter_io_list, 0.25, 0, True)

In [ ]:
import importlib
importlib.reload(importlib.import_module('Pro.Model.Player_Model'))
from Pro.Model.Player_Model import RNN_Model as Pro_Model
from College.Model.College_Model import RNN_Model as College_Model

import torch
from Constants import device

pro_model = Pro_Model(
    input_size=train_dataset.GetProInputSize(),
    mutators=torch.empty(0),
    data_prep=data_prep.pro_data_prep,
    is_hitter=True,
).to(device)
col_model = College_Model(
    input_size=train_dataset.GetColInputSize(),
    data_prep=data_prep.college_data_prep,
    is_hitter=True,
    output_hidden_size=pro_model.GetHiddenSize(),
    output_num_layers=pro_model.GetNumLayers(),
).to(device)

pro_model.optimizer.param_groups[0]["lr"] = 0.005

In [ ]:
importlib.reload(importlib.import_module('Combined.Model.Model_Train'))
from Combined.Model.Model_Train import TrainAndGraph

best_loss = TrainAndGraph(
    pro_network=pro_model,
    col_network=col_model,
    train_dataset=train_dataset,
    test_dataset=test_dataset,
    pro_model_name="../Models/test_pro_hit",
    col_model_name="../Models/test_col_hit",
    is_hitter=True
)

print(best_loss)